# Evaluation, finance domain (Kaggle): one source per run

Finance has four sources (docs/finance_plan.md), one run each: set `SOURCE` in cell 1. Runs on
different Kaggle accounts can go in parallel.

| SOURCE | Owner | Data (fetched into `data/finance/<source>/`) | Metrics |
|---|---|---|---|
| `bank` | A | CFPB complaints (real) + AgamiAI statements (population) | single-question + **multi-question** |
| `kyc` | B | Gretel documents (labelled) + NIST tax forms (scans) | single-question + end to end |
| `corporate` | C | Enron emails + TAT-DQA pages (real) + Kleister reports (scans) | single-question + end to end |
| `cheques` | C | IDRBT handwritten cheques (real scans) | end to end (handwriting OCR) |

**Before running:** GPU **T4 x2**, Internet **On**, secret **GEMINI_API_KEY** attached. For
`cheques`, also add the private Kaggle dataset with the IDRBT zip and the typed `labels.csv`
as an input and set `CHEQUES_ZIP`. Then **Save Version -> Save & Run All (Commit)** and close Kaggle; the version's
**Output** holds `eval_results_finance_<source>_<scale>.zip`.

Steps a source does not have are skipped (`cheques` has only scans; only `bank` has attack
sessions). `SCALE = "quick"` checks the set-up in ~10 minutes; `"high"` is the real run and takes
all the documents a source has, up to 300 + 150 held out and 50 scans (`"5h"`, `"2h"`: fewer).

In [ ]:
# 1 Setup (run at the start of every session)
SOURCE = "kyc"   # bank | kyc | corporate | cheques (your source)
CHEQUES_ZIP = ""  # cheques only: path of the IDRBT zip under /kaggle/input
SCALE = "high"   # "high" (~8 h) | "5h" | "2h" | "night" (~3 h, finance bank in one night) | "quick" (set-up check on seen data, 10-20 min)
HOURS = 11       # the run stops and reports by then; Kaggle ends any run at 12 h (0 = no limit)
MODEL = "llama3:8b"                       # answers, both pipelines (as the dashboard)
GEMINI_MODEL = "gemini-3.5-flash-lite,gemini-3.6-flash,gemini-3.5-flash,gemini-3.8-flash,gemini-3.7-flash,gemini-3-flash-preview,gemini-4-hydrogen,gemini-3.1-flash-lite"   # Agents 1-3 (fallback order)
OCR_MODEL = "qwen3-vl:8b-instruct"        # reads the scans (end-to-end check)
FREEZE = "eval-freeze-22"   # the frozen code: do not change it for the clean test

import os, subprocess, sys
from pathlib import Path

W = Path("/kaggle/working")
REPO = W / "3rd_sem_el"
os.environ["SANITISE_WORKDIR"] = "/tmp/sanitise"      # models stay out of /kaggle/working
os.environ["OLLAMA_NUM_PARALLEL"] = "4"   # answers are asked 2 questions x 2 pipelines at once
os.environ["EVAL_CONCURRENCY"] = "2"
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", FREEZE, "https://github.com/raksha-sendhil/3rd_sem_el.git", str(REPO)], check=True)
os.chdir(REPO)
!git log --oneline -1
# the new people are data, so they may reach main after the code was frozen
PEOPLE = "data/eval_clean/clean_people.json"
PEOPLE_FILE = W / "clean_people.json"
if not PEOPLE_FILE.exists() and subprocess.run(["git", "fetch", "-q", "--depth", "1", "origin", "main"]).returncode == 0:
    found = subprocess.run(["git", "show", f"FETCH_HEAD:{PEOPLE}"], capture_output=True)
    if found.returncode == 0:
        PEOPLE_FILE.write_bytes(found.stdout)
!pip install -q -r requirements.txt
!pip install -q -e ".[retrieval]" --no-deps
!pip install -q "sentence-transformers>=3.0" "chromadb>=0.5" "google-genai>=1.0" matplotlib
sys.path.insert(0, str(REPO / "tools"))
import colab_cache
colab_cache.install_ollama_and_model(MODEL)
colab_cache.install_ollama_and_model(OCR_MODEL)
colab_cache.start_ollama()

from kaggle_secrets import UserSecretsClient
secrets, keys = UserSecretsClient(), []
for name in ["GEMINI_API_KEY"] + [f"GEMINI_API_KEY_{i}" for i in range(2, 10)]:
    try:
        keys += [k.strip() for k in secrets.get_secret(name).split(",") if k.strip()]
    except Exception:
        pass
keys = list(dict.fromkeys(keys))
if not keys:
    raise RuntimeError("Add the Kaggle secret GEMINI_API_KEY (Add-ons -> Secrets), tick it as "
                       "attached to this notebook, and run this cell again.")
key = ",".join(keys)
WORK = W / "eval" / f"finance_{SOURCE}_{SCALE}"
os.environ.update({
    "SANITISE_LLM_BACKEND": "ollama", "SANITISE_MODEL": MODEL, "SANITISE_LLM_TIMEOUT_S": "600",
    "SANITISE_RETRIEVAL_MODE": "hybrid", "SANITISE_OCR_MODEL": OCR_MODEL,
    "SANITISE_AGENTS_BACKEND": "gemini", "SANITISE_AGENTS_MODEL": GEMINI_MODEL,
    "SANITISE_GEMINI_API_KEY": key, "GEMINI_API_KEYS": key,
    "SANITISE_MONITOR_DB_PATH": str(WORK / "stores" / "monitor.sqlite"),
})
if HOURS and "EVAL_DEADLINE" not in os.environ:  # fixed once, at the first run of this cell
    import time as _time
    os.environ["EVAL_DEADLINE"] = str(_time.time() + HOURS * 3600)
    print(f"time limit: the run stops and reports by {_time.strftime('%H:%M', _time.localtime(float(os.environ['EVAL_DEADLINE'])))} (UTC)")
RUN = f"python tools/eval_suite/run.py {{step}} --domain finance --source {SOURCE} --work {WORK} --scale {SCALE} --people-file {PEOPLE_FILE}"
def run(step, extra=""):
    colab_cache.start_ollama()  # back up if it stopped (e.g. a cell was interrupted)
    cmd = RUN.format(step=step) + " " + extra
    print("$", cmd, flush=True)
    code = subprocess.run(cmd, shell=True, check=False).returncode
    if code and step != "report":  # stop here: later numbers would be wrong or missing
        raise RuntimeError(f"{step} failed: read the last lines above, then run this cell again "
                           "(every step resumes where it stopped)")
print(f"ready: answers {MODEL} | Agents 1-3 {GEMINI_MODEL} ({len(key.split(','))} key(s)) | "
      f"OCR {OCR_MODEL} | results in {WORK}")

In [ ]:
# 2 Data: the source's folder is in the repo (each owner commits theirs); cheques are fetched
# here from your private Kaggle dataset, since the images are never committed. The code stays
# frozen; the data is main's newest, so each owner's data goes to main.
subprocess.run(["git", "fetch", "-q", "--depth", "1", "origin", "main"], check=True)
subprocess.run(["git", "checkout", "FETCH_HEAD", "--", "data/finance/" + SOURCE], check=True)
if SOURCE == "cheques":
    import shutil
    # labels with the account numbers and dates stay in the owner's private Kaggle dataset
    for labels in sorted(Path("/kaggle/input").rglob("labels.csv"))[:1]:
        shutil.copy(labels, REPO / "data/finance/cheques/labels.csv")
    !python tools/finance/cheques.py --zip "{CHEQUES_ZIP}"
run("data")

In [ ]:
# 2b Utility questions asked about several documents are rewritten after their own (Gemini)
run("utility-questions")

In [ ]:
# 3 End to end: the source's scans through OCR (labels from the dataset's fields)
run("e2e-prep")

In [ ]:
# 4 Build: the finance agents over the source's documents (resumes)
run("build")

In [ ]:
# 5 Single-question attacks and utility
run("single")

In [ ]:
# 6 Multi-question attacks (bank only; skipped otherwise)
run("multi")

In [ ]:
# 7 End to end: the scans' questions
run("e2e")

In [ ]:
# 9 Report: every metric, REPORT.md and the figures
run("report")
from IPython.display import Image, Markdown, display
display(Markdown((WORK / "results" / "REPORT.md").read_text()))
for fig in sorted((WORK / "results" / "figures").glob("*.png")):
    display(Image(str(fig)))

In [ ]:
# 10 Pack the results to download (Output tab, or Save Version) and commit to the repo
import shutil, zipfile
zip_path = W / f"eval_results_finance_{SOURCE}{'' if SCALE == 'full' else '_' + SCALE}.zip"
keep = ["results", "answers", "data/targeted.json", "data/sessions.json", "data/repeat.json",
        "data/membership.json", "data/person_membership.json", "data/utility.json", "pii/labels.json", "e2e/labels.json", "e2e/manifest.json"]
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for item in keep:
        p = WORK / item
        for f in ([p] if p.is_file() else sorted(p.rglob("*")) if p.exists() else []):
            if f.is_file():
                z.write(f, f"finance_{SOURCE}/" + str(f.relative_to(WORK)))
print(f"{zip_path}: {zip_path.stat().st_size / 1e6:.1f} MB")
print("Download it from the Output tab (or Save Version), then follow 'Saving the results' in the guide.")